# RQ1: transfer — results

Does continued pretraining (CPT) on credit tables improve prediction on credit tables the model has never seen? The notebook follows the paper's argument: the pre-specified test, together with what CPT costs on non-credit tables and how much of any change is specific to credit data (section 1); how the effect varies over tables and seeds (section 2) and over aspects of prediction quality (section 3); and what a practitioner should do instead or in addition (sections 4-7). Five-fold evaluation of the reference recipe (E1) against the original models on identical context and test rows. The pre-specified primary comparison comes first; the rest qualifies it: per-table effects and seed noise (E2), the full metric profile including credit-scoring measures (KS, H-measure, expected maximum profit), the position relative to tuned classical baselines, the matched non-credit control (E4), fine-tuning on the target table (E5), a TabArena-style leaderboard of every method and training precision (E3).

Unit of analysis: one value per held-out table (folds are averaged first; only tables with all five paired folds enter). Intervals come from a hierarchical bootstrap over tables and their folds.

In [1]:
%matplotlib inline
import sys
from pathlib import Path
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import pandas as pd
from src.visualize import style, campaign as cp, campaign2 as c2, results2 as r2
from src.visualize.figures import FigureSaver
style.apply()
sink = FigureSaver('rq1/02_results')
report = cp.NotebookReport('RQ1: transfer — results', sink=sink)
gains = {track: c2.endpoint_gains('reference', track) for track in ('pd', 'lgd')}
seeds = {track: c2.seed_noise(track) for track in ('pd', 'lgd')}
outcomes = c2.primary_spec()['outcomes']

## 1. Primary comparison (pre-specified), forgetting and domain specificity

Question: does the reference CPT model (E1, seed 42) beat the original model on the held-out credit tables? Fixed before any result was opened (`config/design.yaml`, `analysis.primary`): per task a ranking measure and a proper score (PD: AUC and Brier score; LGD: R² and CRPS), per base model; Holm adjustment over all 16 comparisons. No equivalence margin is pre-specified, so a comparison that is not significant is reported with its interval, not called equivalent. The same figure carries two secondary estimates that qualify the test: the change on the non-credit retention tables (forgetting) and the difference between credit CPT and CPT on the matched non-credit twins (E4; the credit-specific part). All other sections are secondary and descriptive.

In [2]:
primary = c2.primary_endpoint()
retention = pd.concat([c2.outcome_intervals('reference', track, metrics) for track, metrics in outcomes.items()],
                      ignore_index=True)
contrast = c2.domain_contrast_intervals(outcomes)
fig = r2.primary_forest(primary, retention, contrast, title='Pre-specified comparison: CPT (E1) against the original model')
cp.show(sink, c2.pages(fig, 'primary_endpoint',
    'Five-fold improvement of the reference CPT model (E1, seed 42) over the original model, per base model (rows) and '
    'pre-specified outcome (panels): AUC and Brier score for PD, R² and CRPS for LGD, oriented so that positive is '
    'better (a decrease in Brier score and CRPS counts as positive). Filled: equal-table mean over the held-out credit '
    'tables; hollow: over non-credit TabArena retention tables. Only tables with five finite paired folds contribute. Planned pools: 17 PD, 8 LGD and 19 binary, 12 regression retention tables. '
    'squares: credit CPT (E1) minus CPT with the same recipe on the matched non-credit twins of the training tables (E4), '
    'on the held-out credit tables and identical rows. Lines: '
    '95% percentile intervals of a hierarchical bootstrap that resamples tables and, within each, its five folds '
    '(2,000 draws). Asterisk: Holm-adjusted bootstrap p < 0.05 over the 16 comparisons (tasks × outcomes × models).'))
report.table('Primary comparison (Holm over 16 tests)', primary)
report.table('Same outcomes on credit and retention tables', retention)
report.table('Credit CPT minus non-credit CPT (E1 − E4) per outcome', contrast)
report.add('1. Primary comparison (pre-specified)',
           'E1 has no complete primary comparison in the local results.' if primary.empty else
           f'{primary.p_value.notna().sum()} of 16 planned comparisons have at least one table with five paired folds.')

No measurements available for this section yet.


""


""


""


## 2. Per-table transfer and seed noise (E1, E2)

Question: on which held-out credit tables does CPT change the five-fold score, by how much, and is the change larger than retraining with another seed moves it?

In [3]:
for track in ('pd', 'lgd'):
    fig = r2.table_gains(gains[track], seeds[track], track=track,
                         title='Effect of CPT per held-out credit table (E1, E2)')
    cp.show(sink, c2.pages(fig, 'reference_tables',
        'Five-fold change of ' + r2.SCORE[track] + ' from continued pretraining with the reference recipe, per held-out '
        'credit table (rows) and base model (columns): CPT minus the original model on identical context and test rows, '
        'averaged over the five outer folds (seed 42, E1). Horizontal lines span the changes of seeds 42, 43 and 44 '
        '(E1, E2); a filled marker means all three seeds agree in sign and the smallest absolute change exceeds their '
        'range. The dashed line is the equal-table mean of seed 42. Only tables with all five paired folds are shown; '
        'horizontal scales differ between panels.'), track=track)
    report.table(track.upper() + ' E1 summary per model (credit and retention tables)', c2.gain_summary(gains[track]))
    report.table(track.upper() + ' E1 per-table gains', gains[track])
    report.table(track.upper() + ' E2 summary per seed arm', c2.gain_summary(c2.endpoint_gains('seed', track)))
    report.table(track.upper() + ' gains at three seeds', seeds[track])
report.add('2. Per-table transfer and seed noise (E1, E2)', c2.evidence_note('E1 five-fold gains', gains['pd']))

No measurements available for this section yet.


""


""


""


""


No measurements available for this section yet.


""


""


""


""


## 3. Ranking, profit, calibration and distribution quality

Question: does CPT change probability calibration (PD) or the predictive distribution (LGD) differently from ranking, and does a ranking change matter in money? PD adds the credit-scoring measures of Lessmann et al. (2015): KS, the H-measure and the expected maximum profit (EMP; Verbraken et al. 2014, per applicant as a fraction of the loan). Each metric is oriented so that a positive value is an improvement.

For the TabPFN LGD models, negative log likelihood (NLL) scores the observed loss under the predicted density. The tables show the uncapped NLL and, for sensitivity, the score obtained with the historical clipping rule. The clipping count includes every nonfinite loss and every finite loss outside [-100, 100]. The nonfinite count measures numerical failures alone. A table with any failed row has an unavailable uncapped NLL. Counts and NLL pool held-out rows across its five folds. The model summary gives each table equal weight and also reports the failure and clipping rates pooled over all rows.

In [4]:
metrics = {track: c2.endpoint_metrics('reference', track) for track in ('pd', 'lgd')}
profiles = {track: c2.select(c2.metric_profile(metrics[track]), arm=0) for track in ('pd', 'lgd')}
fig = r2.metric_profile_heatmap(profiles, title='Which aspects of prediction quality CPT changes (E1)')
cp.show(sink, c2.pages(fig, 'metric_profile',
    'Five-fold effect of continued pretraining with the reference recipe (E1) on discrimination and on probability or '
    'distribution quality, per base model (rows) and metric (columns), over the held-out credit tables with all five '
    'paired folds. Colour: share of tables on which CPT improved the metric; text: number of improved tables out of '
    'the tables compared (the equal-table mean improvements are listed in the table below the figure). Improvements are oriented so that positive is better: increase in AUC, KS, '
    'H-measure (Beta(2, 2) costs), expected maximum profit (EMP, per applicant as a fraction of the loan; LGD of a '
    'rejected defaulter 0 with probability 0.55, 1 with probability 0.10, uniform otherwise; ROI 0.2644), R² and '
    'Spearman ρ; decrease in log loss, Brier score, ECE, MAE and CRPS; decrease in |calibration slope − 1|, '
    '|calibration-in-the-large|, |mean bias| and |prediction SD / target SD − 1|.'))
for track in ('pd', 'lgd'):
    report.table(track.upper() + ' metric profile (credit tables)', profiles[track])
    report.table(track.upper() + ' metric profile (retention tables)', c2.select(c2.metric_profile(metrics[track], credit=False), arm=0))
density_by_table, density_summary = c2.lgd_density_tables()
report.table('LGD TabPFN density scores and failure rates by model and start', density_summary)
report.table('LGD TabPFN density scores, counts and rates on each held-out credit table', density_by_table)
report.add('3. Ranking, profit, calibration and distribution quality', c2.evidence_note('E1 metric profile', profiles['pd']))

No measurements available for this section yet.


""


""


""


""


""


""


## 4. Position relative to tuned classical baselines

Question: where do the original and the CPT models stand relative to tuned CatBoost and XGBoost (50 Optuna trials each) on the same folds, and does CPT change that position? Reported per table; the leaderboard of section 7 places every method, fine-tuned ones included, in one ranking.

In [5]:
baselines = {track: c2.baseline_comparison(track) for track in ('pd', 'lgd')}
for track in ('pd', 'lgd'):
    report.table(track.upper() + ' scores relative to CatBoost per table (with the XGBoost and linear baselines)', baselines[track])
report.add('4. Position relative to tuned classical baselines', c2.evidence_note('baseline comparison', baselines['pd']))

""


""


## 5. Credit versus matched non-credit CPT (E4)

Question: is the transfer domain-specific? Each held-out credit table is scored after CPT on the credit training tables (E1) and after CPT on their matched non-credit twins (E4), on the same folds and rows. The difference with its interval is drawn in the figure of section 1 (squares); the tables below give it per table.

In [6]:
controls = {track: c2.domain_control_gains(track) for track in ('pd', 'lgd')}
for track in ('pd', 'lgd'):
    report.table(track.upper() + ' E4 versus E1 per table', controls[track])
    report.table(track.upper() + ' E4 versus E1 per model', controls[track].groupby('model')[['gain_credit', 'gain_noncredit', 'credit_minus_noncredit']].agg(['mean', 'median', 'size']).reset_index() if not controls[track].empty else controls[track])
control_intervals = {track: c2.domain_control_intervals(track) for track in ('pd', 'lgd')}
for track in ('pd', 'lgd'):
    report.table(track.upper() + ' credit minus non-credit CPT gain with 95% bootstrap interval', control_intervals[track])
report.add('5. Credit versus matched non-credit CPT (E4)', c2.evidence_note('E4 domain control', controls['pd']))

""


""


""


""


""


""


## 6. CPT versus fine-tuning on the target table (E5)

Question: what should a practitioner with a new credit table do — use the original model, a CPT model, fine-tune on the table, or both — and does the answer depend on how many labelled rows the table has? All four conditions share the context cap, folds and test rows. The table below the figure is the recommendation: per task, labelled-row band and model, the mean change of each option, how often each scored best, and its GPU cost (CPT paid once, fine-tuning per table).

In [7]:
results = c2.task_results('finetune')
adaptation = c2.adaptation_by_size(results)
fig = r2.adaptation_by_size(adaptation, title='Continued pretraining, target fine-tuning and both, by table size (E5)')
cp.show(sink, c2.pages(fig, 'target_finetune',
    'Five-fold change per held-out credit table and base model (colour) against the table\'s labelled context rows '
    '(the fold\'s training rows after a 20% validation split; log scale), for the three choices of E5 (rows): continued '
    'pretraining alone (CPT, E1) against the original model; the original model fine-tuned on the fold\'s training '
    'rows (FT) against the original model; and the CPT model fine-tuned the same way (CPT+FT) against FT. Fine-tuning '
    'uses the upstream fine-tuners with the learning rate chosen from 1e-6, 1e-5 and 6.5e-5 by validation AUC '
    '(PD) or RMSE (LGD), at most 30 epochs, early stopping after 8 epochs without improvement. AUC for PD, R² for LGD; '
    'all conditions share the context cap, validation and test rows; tables with all five folds only.'))
cost = c2.cost_comparison(c2.run_resources(c2.load_trajectories(['reference'])), results)
report.table('E5 recommendation: change per option, best option and cost, by labelled-row band', c2.adaptation_summary(adaptation, cost))
report.table('E5 summary per model and start', c2.finetune_summary(results))
report.table('E5 per table', adaptation)
report.table('GPU hours for one reusable E1 checkpoint and one E5 target fit including rate selection. Campaign totals include four partitions for E1 and five folds per target for E5. Break-even requires full coverage on the same GPU model.', cost)
report.add('6. CPT versus fine-tuning on the target table (E5)', c2.evidence_note('E5 fine-tuning', adaptation))

No measurements available for this section yet.


""


""


""


""


## 7. Leaderboard of all methods

Question: how do the original models, CPT, fine-tuning on the table, CPT followed by fine-tuning and the tuned classical baselines rank against each other when every method meets every other on the same folds? Computed as on TabArena: Elo from pairwise wins per (table, fold), with every table weighted equally and the linear baseline anchored at 1000; the normalized score (best method of a table 1, median method 0); and improvability (relative error gap to the best method of each fold). Elo, rank and win rate count wins regardless of their size, so they complement the effect sizes above rather than replace them.

In [8]:
boards = {track: c2.leaderboard(track, results) for track in ('pd', 'lgd')}
for track in ('pd', 'lgd'):
    fig = r2.leaderboard(boards[track]['board'], track=track, title='Leaderboard of every method on the held-out credit tables')
    cp.show(sink, c2.pages(fig, 'leaderboard',
        'Leaderboard of every method scored on the same outer folds of the held-out credit tables (' + r2.TRACK_NAME[track] + '; '
        'error ' + c2.LEADERBOARD_ERROR[track][1] + '), following the TabArena evaluation code. The foundation models, with '
        'and without fine-tuning, are scored on the rows of E5 under one context cap (at most 50,000 labelled rows; TabPFN v2 '
        '10,000), so fine-tuning is not compared with a model given more context; the tuned baselines use the full training rows. Left: Elo from a '
        'Bradley-Terry fit to the pairwise wins in each (table, fold), ties counted as half a win and every table weighted '
        'equally, anchored at ' + c2.LEADERBOARD_ANCHOR[track].lower() + ' = 1000 (dotted); lines are 95% intervals from '
        '200 bootstrap resamples of the tables. Middle: normalized score, the fold-mean error of each table placed between '
        'the best method (1) and the median method (0) of that table, clipped to [0, 1] and averaged over tables. Right: '
        'improvability, one minus the best error divided by the method\'s error in each fold, averaged over folds and tables, '
        'with a 95% interval from 100 table resamples. Colour: base model (classical baselines in grey and brown); marker: '
        'original model (hollow circle), CPT, E1 (filled circle), fine-tuned on the table, E5 (hollow square), CPT then '
        'fine-tuned (filled square), tuned baseline (triangle). Only tables on which every method has every fold.'), track=track)
    report.table(track.upper() + ' leaderboard', boards[track]['board'])
    report.table(track.upper() + ' leaderboard coverage per method (entered: scored on at least half of the tables)', boards[track]['coverage'])
    report.table(track.upper() + ' tables left out of the leaderboard (a method lacked a fold)', pd.DataFrame({'table': boards[track]['dropped']}))
report.add('7. Leaderboard of all methods', c2.evidence_note('leaderboard', boards['pd']['board']))


No measurements available for this section yet.


""


""


,table


No measurements available for this section yet.


""


""


,table


## 8. Training precision (E3)

Does changing training precision alter the effect of CPT on the same data with the same update budget and seed? E3 compares BF16 for TabPFN v2.6 and v3 and strict FP32 for TabICLv2 against each model's E1 reference. E1 uses FP16 autocast with loss scaling and keeps the TabICLv2 column embedder in FP32. TabPFN v2 PD uses FP32 transformer normalizations in E1. Its precision comparison belongs to Experiment 0.

In [9]:
for track in ('pd', 'lgd'):
    precision = c2.arms_overview(['reference', 'precision'], track).rename(
        columns={'training_precision': 'configured_training_precision'})
    report.table(track.upper() + ' five-fold effect per training precision and model (E1 reference and E3 alternatives)', precision)
    report.table(track.upper() + ' E3 mean gain with 95% bootstrap interval', c2.gain_intervals('precision', track))
    report.table(track.upper() + ' E3 per-table gains', c2.endpoint_gains('precision', track))
report.add('8. Training precision (E3)', c2.evidence_note('E3 results', c2.endpoint_gains('precision', 'pd')))

""


""


""


""


""


""


## Summary

Every displayed table and figure, in section order.

In [10]:
print(report.summary(sink))

RQ1: transfer — results

1. Primary comparison (pre-specified)
E1 has no complete primary comparison in the local results.

No measurements available for this figure group.

Table: Primary comparison (Holm over 16 tests)
CSV (first column: row index)
row

Table: Same outcomes on credit and retention tables
CSV (first column: row index)
row

Table: Credit CPT minus non-credit CPT (E1 − E4) per outcome
CSV (first column: row index)
row

2. Per-table transfer and seed noise (E1, E2)
No results yet for E1 five-fold gains; the figure is drawn once they exist.

No measurements available for this figure group.

Table: PD E1 summary per model (credit and retention tables)
CSV (first column: row index)
row

Table: PD E1 per-table gains
CSV (first column: row index)
row

Table: PD E2 summary per seed arm
CSV (first column: row index)
row

Table: PD gains at three seeds
CSV (first column: row index)
row

No measurements available for this figure group.

Table: LGD E1 summary per model (credit and